# Task 2.1 – Khảo sát schema bốn nguồn
Chạy từ repo hoặc thư mục notebooks. Script chỉ đọc raw và xuất tài liệu khảo sát; chưa cleaning.

In [1]:
from pathlib import Path
import importlib.util
from IPython.display import display
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'scripts' / 'schema_survey.py').is_file() and (p / 'data' / 'raw').is_dir()), None)
if ROOT is None: raise FileNotFoundError('Không tìm thấy repo có scripts/schema_survey.py và data/raw.')
spec = importlib.util.spec_from_file_location('schema_survey', ROOT / 'scripts' / 'schema_survey.py')
module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(module)
summary, key_checks, link_checks, findings = module.survey()
print('Đã khảo sát', len(summary), 'trường và xuất docs/source_schema_summary.csv.')

Đã khảo sát 103 trường và xuất docs/source_schema_summary.csv.


## Tên cột, dtype ban đầu, số dòng/cột, NULL, giá trị mẫu và phạm vi ngày
Dtype quan sát phụ thuộc phiên bản Pandas; không dùng dtype của cột toàn NULL để quyết định kiểu đích.

In [2]:
display(summary)
display(summary.loc[summary.date_min != '', ['source', 'column', 'date_min', 'date_max']])
assert len(summary) == 103

,source,file_name,column,dtype_initial,rows_in_source,columns_in_source,null_count,null_pct,unique_non_null,sample_values,date_min,date_max
0,Listings,listings.csv.gz,id,int64,3208,90,0,0.0000,3208,71609 | 71896 | 71903,,
1,Listings,listings.csv.gz,listing_url,object,3208,90,0,0.0000,3208,https://www.airbnb.com/rooms/71609 | https://w...,,
2,Listings,listings.csv.gz,scrape_id,int64,3208,90,0,0.0000,1,20260923225116,,
3,Listings,listings.csv.gz,last_scraped,object,3208,90,0,0.0000,2,2026-09-26 | 2026-09-25,2026-09-25,2026-09-26
4,Listings,listings.csv.gz,source,object,3208,90,1470,45.8229,1,previous scrape,,
...,...,...,...,...,...,...,...,...,...,...,...,...
98,Reviews,reviews.csv.gz,reviewer_id,int64,46000,6,0,0.0000,38863,1456140 | 1804182 | 3113461,,
99,Reviews,reviews.csv.gz,reviewer_name,object,46000,6,0,0.0000,17583,[Không xuất văn bản cá nhân/mô tả dài],,
100,Reviews,reviews.csv.gz,comments,object,46000,6,17,0.0370,39417,[Không xuất văn bản cá nhân/mô tả dài],,
101,Neighbourhoods,neighbourhoods.csv,neighbourhood_group,object,55,2,0,0.0000,5,Central Region | East Region | North-East Region,,


,source,column,date_min,date_max
3,Listings,last_scraped,2026-09-25,2026-09-26
47,Listings,price_quote_checkin_date,2026-09-26,2027-09-23
48,Listings,price_quote_checkout_date,2026-09-27,2027-09-26
66,Listings,calendar_last_scraped,2026-09-25,2026-09-26
74,Listings,first_review,2011-05-04,2026-09-21
75,Listings,last_review,2015-06-08,2026-09-25
91,Calendar,date,2026-09-25,2027-09-25
97,Reviews,date,2011-05-04,2026-09-25


## Grain, khóa ứng viên và liên kết
Các khóa này được khảo sát trên một snapshot; chưa tự động áp ràng buộc warehouse.

In [3]:
display(key_checks)
display(link_checks)
for finding in findings:
    print(finding)

Reviews.id: 41,191 mã khác nhau / 46,000 dòng; 4,809 dòng lặp ID sau lần đầu. Khóa (listing_id,id) có 0 dòng trùng dư trong snapshot này; cần xác minh ngữ nghĩa nguồn trước DDL.
Listings.neighbourhood có 3,208 NULL; dùng cặp neighbourhood_group_cleansed/neighbourhood_cleansed để nối khu vực.
Danh mục có 13 khu vực không có listing; không tự coi là lỗi.
Listings.price và price_quote_price_per_night gần nghĩa nhưng không mặc định thay thế nhau; kiểm tra currency và chênh lệch trước ETL.
Calendar hiện không có cột price. Reviews không có điểm rating; review_scores_rating lấy từ Listings.
Các cột toàn NULL cần ghi nhận và quyết định phạm vi sử dụng, không suy kiểu nghiệp vụ từ float64.
Không JOIN Calendar và Reviews ở mức chi tiết để tính KPI vì hai bảng đều nhiều dòng trên một listing.
snapshot_date lấy từ docs/source_inventory.csv; không đồng nhất với last_scraped hoặc ngày phát sinh review.


,source,grain,candidate_key,rows,missing_key_rows,duplicate_excess_rows
0,Listings,listing/snapshot,id,3208,0,0
1,Calendar,listing/ngày/snapshot,listing_id+date,1170950,0,0
2,Reviews,bản ghi review (khóa ứng viên listing_id + id),listing_id+id,46000,0,0
3,Neighbourhoods,khu vực,neighbourhood_group+neighbourhood,55,0,0


,relationship,missing_key_rows,orphan_rows
0,Calendar.listing_id -> Listings.id,0,0
1,Reviews.listing_id -> Listings.id,0,0
2,"Listings.(group_cleansed, neighbourhood_cleans...",0,0


## Sơ đồ quan hệ nguồn sơ bộ
```mermaid
erDiagram
    NEIGHBOURHOODS ||--o{ LISTINGS : "group + neighbourhood_cleansed"
    LISTINGS ||--o{ CALENDAR : "id = listing_id"
    LISTINGS ||--o{ REVIEWS : "id = listing_id"
```

Khóa ứng viên: Listings(id); Calendar(listing_id,date); Reviews(listing_id,id); Neighbourhoods(neighbourhood_group,neighbourhood). Khi giữ nhiều snapshot, xác định snapshot strategy trước DDL. Đây là quan hệ nguồn, chưa phải star schema cuối cùng.

## Kết luận và bàn giao
Các nhận xét được sinh từ raw ở cell trên và ghi trong docs/schema_survey_findings.md. Không xóa bản ghi, không sửa ID hoặc tự hợp nhất dữ liệu. Ảnh relationship_diagram.png là bản minh họa cũ; ưu tiên grain và khóa ứng viên trong bảng kiểm tra mới khi thiết kế warehouse.